# 01. Data Exploration, Curation & Tokenization Dynamics
**Project:** From Small GPT to an Agentic AI System  
**Objective:** Curate an algorithmic Python corpus, analyze sequence length distributions, and compare Character vs. BPE tokenization compression ratios (RQ2). Compatible with Google Colab (free GPU/CPU).

In [ ]:
# Setup & Environment Check
!pip install -q tokenizers transformers matplotlib seaborn
import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 1. Corpus Generation
We generate and inspect clean algorithmic Python functions covering sorting, search, graph traversal, and dynamic programming.

In [ ]:
import sys
from pathlib import Path
if not Path("src").exists():
    !git clone https://github.com/example/small-gpt-agent.git . 2>/dev/null || true

from src.data.dataset import generate_synthetic_code_corpus, build_and_split_corpus
corpus = generate_synthetic_code_corpus(num_samples=1000, seed=42)
train_set, val_set, test_set = build_and_split_corpus(corpus, train_ratio=0.85, val_ratio=0.075)

print(f"Dataset split: Train={len(train_set)} | Val={len(val_set)} | Test={len(test_set)}")
print("\n--- Sample Corpus Snippet ---")
print(corpus[0][:300] + "\n...")

## 2. Tokenizer Comparison: Character vs. BPE
We train Character and Byte-Level BPE tokenizers to measure compression efficiency.

In [ ]:
from src.data.tokenizer import CharacterTokenizer, BPETokenizerWrapper

# 1. Character Tokenizer
char_tok = CharacterTokenizer().fit(corpus)

# 2. BPE Tokenizers
bpe_500 = BPETokenizerWrapper(vocab_size=500).train_from_iterator(corpus)
bpe_2048 = BPETokenizerWrapper(vocab_size=2048).train_from_iterator(corpus)

sample_doc = "\n".join(test_set[:20])
raw_bytes = len(sample_doc.encode('utf-8'))

char_tokens = char_tok.encode(sample_doc)
bpe_500_tokens = bpe_500.encode(sample_doc)
bpe_2048_tokens = bpe_2048.encode(sample_doc)

print(f"Raw Byte Size: {raw_bytes} bytes")
print(f"Char Tokens:   {len(char_tokens):<6} | Compression: {raw_bytes/len(char_tokens):.2f} bytes/tok | Vocab: {char_tok.vocab_size}")
print(f"BPE-500:       {len(bpe_500_tokens):<6} | Compression: {raw_bytes/len(bpe_500_tokens):.2f} bytes/tok | Vocab: {bpe_500.vocab_size}")
print(f"BPE-2048:      {len(bpe_2048_tokens):<6} | Compression: {raw_bytes/len(bpe_2048_tokens):.2f} bytes/tok | Vocab: {bpe_2048.vocab_size}")

## 3. Visualizing Compression & Sequence Length Reduction

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
tok_names = ["Character", "BPE-500", "BPE-2048"]
ratios = [raw_bytes/len(char_tokens), raw_bytes/len(bpe_500_tokens), raw_bytes/len(bpe_2048_tokens)]
ax.bar(tok_names, ratios, color=["#4A90E2", "#50E3C2", "#B8E986"], edgecolor="black")
ax.set_ylabel("Compression Ratio (Bytes / Token)")
ax.set_title("Tokenizer Compression Efficiency on Python Code (RQ2)")
ax.grid(True, linestyle="--", alpha=0.6)
plt.show()